# Описание задачи

Командный ШИФТ Интенсив Classical ML (2025). Результат: 3 место из 8 команд

**Инференс модели для задачи регрессии - предсказание остатка на счетах клиента через 2 месяца**



### Установка библиотек и загрузка данных

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [ ]:
!pip install catboost
from catboost import CatBoostRegressor

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.1 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive') #вылезет регистрация
# Путь к папке с ноутбуком
notebook_path = '/content/drive/MyDrive/доп прога/Шифт интенсив'
# Чтение файла
target = pd.read_csv(os.path.join(notebook_path, 'train_target.csv'))
# Путь к папке с ноутбуком2
notebook_path2 = '/content/drive/MyDrive/доп прога/Шифт интенсив/data'
# Чтение файла
maindf = pd.read_parquet(os.path.join(notebook_path2, 'train_main_df.parquet'))
main_test = pd.read_parquet(os.path.join(notebook_path2, 'test_main_df.parquet'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Обучение модели

In [ ]:
feat7 = ['savings_sum_sa_now', 'savings_sum_sa_1m', 'savings_sum_sa_2m', 'savings_sum_sa_3m',
        'savings_avg_sa_3m', 'savings_sum_sa_credit_3m', 'savings_sum_sa_6m']

feat6 = ['savings_sum_sa_now', 'savings_sum_sa_1m', 'savings_sum_sa_2m', 'savings_sum_sa_3m',
        'savings_avg_sa_3m', 'savings_sum_sa_credit_3m']

feat5 = ['savings_sum_sa_now', 'savings_sum_sa_1m', 'savings_sum_sa_2m',
        'savings_avg_sa_3m', 'savings_sum_sa_credit_3m']
      #'savings_avg_sa_6m', 'savings_sum_sa_6m','savings_sum_sa_9m','savings_sum_sa_12m',

train_df = maindf[feat6].copy()
test_df = main_test[feat6].copy()

# Разделение на тренировочную и тестовую выборку
y = np.log1p(target['target'])
X_train, X_test, y_train, y_test = train_test_split(train_df, y, test_size=0.1, random_state=42)

In [ ]:
modelcat = CatBoostRegressor(
    bootstrap_type='MVS',          #бутстрап для длинных хвостов
    grow_policy='Lossguide',      # Лучше ловит выбросы
    max_leaves=64,
    l2_leaf_reg=5,
    min_data_in_leaf=20,
    iterations=250,
    learning_rate=0.07,
    has_time=False,  # Если данные не временные
    boosting_type='Plain',  # Лучше для регрессии
    score_function='L2',  # Для регрессии
    random_state=42,
    verbose=20,
    rsm = 0.7,
    max_depth=6
)
modelcat.fit(X_train, y_train,
            eval_set=(X_test, y_test),
            use_best_model=True
)

# Предсказания и RMSE
train_pred = modelcat.predict(X_train)
test_pred = modelcat.predict(X_test)

train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

print(f'Train RMSE: {train_rmse:.4f}')
print(f'Test RMSE: {test_rmse:.4f}')

0:	learn: 5.2858931	test: 5.3149466	best: 5.3149466 (0)	total: 144ms	remaining: 35.8s
20:	learn: 2.4630861	test: 2.4642743	best: 2.4642743 (20)	total: 1.89s	remaining: 20.6s
40:	learn: 2.1747722	test: 2.1764720	best: 2.1764720 (40)	total: 3.56s	remaining: 18.2s
60:	learn: 2.1407259	test: 2.1533774	best: 2.1533774 (60)	total: 4.38s	remaining: 13.6s
80:	learn: 2.1230486	test: 2.1498505	best: 2.1497029 (76)	total: 5s	remaining: 10.4s
100:	learn: 2.1097255	test: 2.1509713	best: 2.1497029 (76)	total: 5.58s	remaining: 8.23s
120:	learn: 2.0985366	test: 2.1511950	best: 2.1497029 (76)	total: 6.12s	remaining: 6.53s
140:	learn: 2.0857394	test: 2.1510161	best: 2.1497029 (76)	total: 6.69s	remaining: 5.17s
160:	learn: 2.0748623	test: 2.1524285	best: 2.1497029 (76)	total: 7.38s	remaining: 4.08s
180:	learn: 2.0638144	test: 2.1529727	best: 2.1497029 (76)	total: 8.79s	remaining: 3.35s
200:	learn: 2.0538058	test: 2.1550048	best: 2.1497029 (76)	total: 10.2s	remaining: 2.48s
220:	learn: 2.0445377	test: 2.1

In [ ]:
#6:    2.1834
#7:    2.1755
#7+id: 2.1741
#8:    2.1685
#9:    2.1711
#10:   2.1283

#8: 2.1380 с признаком savings_sum_sa_credit_3m
#7: 2.1411
#6: 2.1497
#5: 2.1623

# Запуск Gradio

In [ ]:
import gradio as gr

In [ ]:
def predict_balance(sum_now, sum1, sum2, sum3, avg3, credit3):

    input_data = pd.DataFrame([[sum_now, sum1, sum2, sum3, avg3, credit3]],
                              columns=feat6
    )

    prediction = modelcat.predict(input_data)

    return f"Предполагаемый остаток на счете: {np.exp(prediction) - 1}"

In [ ]:
predict_balance(500, 500, 500, 500, 700, 1000)

'Предполагаемый остаток на счете: 86.02712179512969'

In [ ]:
# Описание входных полей
inputs = [
    gr.Slider(0.0, 300000000.0, step=1000, label="Суммарный остаток на текущую дату (руб)"),
    gr.Slider(0.0, 300000000.0, step=1000, label="Остаток 1 мес. назад (руб)"),
    gr.Slider(0.0, 300000000.0, step=1000, label="Остаток 2 мес. назад (руб)"),
    gr.Slider(0.0, 300000000.0, step=1000, label="Остаток 3 мес. назад (руб)"),
    gr.Slider(0.0, 300000000.0, step=1000, label="Средний баланс за 3 последних мес.(руб)"),
    gr.Slider(0.0, 500000000.0, step=1000, label="Сумма поступлений денежных средств на счета за 3 последних мес.(руб)")

]

# Заголовок и примеры
title = "Прогнозирование остатка на счёте"
description = "Введите данные клиента\n"

# Создание интерфейса
demo = gr.Interface(
    fn=predict_balance,
    inputs=inputs,
    outputs="text",
    title=title,
    description=description
)

demo.launch(inbrowser=True) #inline=True)

It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4c43a98e4d8bc6c3dc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
def predict_balance(sum_now, sum1, sum2, sum3, avg3, credit3):
    # Фиксированные названия столбцов
    input_data = pd.DataFrame([[sum_now, sum1, sum2, sum3, avg3, credit3]],
                            columns=['savings_sum_sa_now', 'savings_sum_sa_1m',
                                    'savings_sum_sa_2m', 'savings_sum_sa_3m',
                                    'savings_avg_sa_3m', 'savings_sum_sa_credit_3m'])

    prediction = modelcat.predict(input_data)
    return f"Предполагаемый остаток на счете: {np.expm1(prediction[0]):,.2f} руб".replace(",", " ")

# Обновленные входы с правильным порядком
inputs = [
    gr.Slider(0.0, 300000.0, value=500.0, step=1000, label="Суммарный остаток на текущую дату (руб)"),
    gr.Slider(0.0, 300000.0, value=500.0, step=1000, label="Остаток 1 мес. назад (руб)"),
    gr.Slider(0.0, 300000.0, value=500.0, step=1000, label="Остаток 2 мес. назад (руб)"),
    gr.Slider(0.0, 300000.0, value=500.0, step=1000, label="Остаток 3 мес. назад (руб)"),
    gr.Slider(0.0, 300000.0, value=500.0, step=1000, label="Средний баланс за 3 мес. (руб)"),
    gr.Slider(0.0, 500000.0, value=500.0, step=1000, label="Поступления за 3 мес. (руб)")
]

demo = gr.Interface(
    fn=predict_balance,
    inputs=inputs,
    outputs="text",
    title="Прогнозирование остатка на счёте",
    description="Введите параметры клиента для прогноза"
)

# Для Google Colab
demo.launch(share=True)

# Для локального запуска
# demo.launch()

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2025/07/09 11:41:08 [W] [service.go:132] login to server failed: dial tcp 44.237.78.176:7000: i/o timeout


<IPython.core.display.Javascript object>

# Запуск  Streamlit-интерфейса

In [ ]:
!pip install streamlit scikit-learn pandas

In [ ]:
%%writefile money_streamlit_app.py
# 1. Установка необходимых библиотек (выполнить в терминале)
# pip install streamlit scikit-learn pandas

# 2. Создаем файл money_streamlit_app
import streamlit as st
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
!pip install catboost
from catboost import CatBoostRegressor


import os
from google.colab import drive
drive.mount('/content/drive') #вылезет регистрация
# Путь к папке с ноутбуком
notebook_path = '/content/drive/MyDrive/доп прога/Шифт интенсив'
# Чтение файла
target = pd.read_csv(os.path.join(notebook_path, 'train_target.csv'))
# Путь к папке с ноутбуком2
notebook_path2 = '/content/drive/MyDrive/доп прога/Шифт интенсив/data'
# Чтение файла
maindf = pd.read_parquet(os.path.join(notebook_path2, 'train_main_df.parquet'))
main_test = pd.read_parquet(os.path.join(notebook_path2, 'test_main_df.parquet'))
feat7 = ['savings_sum_sa_now', 'savings_sum_sa_1m', 'savings_sum_sa_2m', 'savings_sum_sa_3m',
        'savings_avg_sa_3m', 'savings_sum_sa_credit_3m', 'savings_sum_sa_6m']

feat6 = ['savings_sum_sa_now', 'savings_sum_sa_1m', 'savings_sum_sa_2m', 'savings_sum_sa_3m',
        'savings_avg_sa_3m', 'savings_sum_sa_credit_3m']

feat5 = ['savings_sum_sa_now', 'savings_sum_sa_1m', 'savings_sum_sa_2m',
        'savings_avg_sa_3m', 'savings_sum_sa_credit_3m']
      #'savings_avg_sa_6m', 'savings_sum_sa_6m','savings_sum_sa_9m','savings_sum_sa_12m',

train_df = maindf[feat6].copy()
test_df = main_test[feat6].copy()

# Разделение на тренировочную и тестовую выборку
y = np.log1p(target['target'])
X_train, X_test, y_train, y_test = train_test_split(train_df, y, test_size=0.1, random_state=42)
modelcat = CatBoostRegressor(
    bootstrap_type='MVS',          #бутстрап для длинных хвостов
    grow_policy='Lossguide',      # Лучше ловит выбросы
    max_leaves=64,
    l2_leaf_reg=5,
    min_data_in_leaf=20,
    iterations=250,
    learning_rate=0.07,
    has_time=False,  # Если данные не временные
    boosting_type='Plain',  # Лучше для регрессии
    score_function='L2',  # Для регрессии
    random_state=42,
    verbose=20,
    rsm = 0.7,
    max_depth=6
)
modelcat.fit(X_train, y_train,
            eval_set=(X_test, y_test),
            use_best_model=True
)

# Предсказания и RMSE
train_pred = modelcat.predict(X_train)
test_pred = modelcat.predict(X_test)

train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

print(f'Train RMSE: {train_rmse:.4f}')
print(f'Test RMSE: {test_rmse:.4f}')



# Функция для предсказания
def predict_species(sum_now, sum1, sum2, sum3, avg3, credit3):
    #maindf[feat6].copy()
    input_data = pd.DataFrame([[sum_now, sum1, sum2, sum3, avg3, credit3]],
                             columns=train_df.feature_names)
    prediction = modelcat.predict(input_data)
    return prediction

# Интерфейс Streamlit
st.title('Прогнозирование остатка на счёте')
st.write('Это приложение прогнозирует остаток на счёте на горизонте 2мес')

# Сайдбар с параметрами
st.sidebar.header('Данные клиента')
sum_now = st.sidebar.slider('Суммарный остаток на текущую дату (руб)', 0.0, 300000000.0, 500.0)
sum1 = st.sidebar.slider('Суммарный остаток 1 мес. назад (руб)', 0.0, 300000000.0, 500.0)
sum2 = st.sidebar.slider('Суммарный остаток 2 мес. назад (руб)', 0.0, 300000000.0, 500.0)
sum3 = st.sidebar.slider('Суммарный остаток 3 мес. назад (руб)', 0.0, 300000000.0, 500.0)
avg3 = st.sidebar.slider('Средний баланс за 3 последних мес.(руб)', 0.0, 300000000.0, 500.0)
credit3 = st.sidebar.slider('Сумма поступлений денежных средств на счета за 3 последних мес.(руб)', 0.0, 500000000.0, 500.0)

# Предсказание
if st.sidebar.button('Составить прогноз'):
    prediction = predict_species(sum_now, sum1, sum2, sum3, avg3, credit3)

    st.subheader('Результат прогноза')
    st.success(f'Остаток на счёте: **{prediction}**')

# Информация о датасете
if st.checkbox('Показать информацию о датасете'):
    st.subheader('Датасет X_train')
    st.write(X_train.DESCR)
    st.write('Первые 5 строк данных:')
    st.write(X_train.head())

Writing money_streamlit_app.py


In [ ]:
!streamlit run money_streamlit_app.py




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.148.78.58:8501

  Stopping...
^C


# Запуск 2

In [ ]:
!pip install streamlit

In [ ]:
!pip install streamlit scikit-learn pandas

In [ ]:
%%writefile money_streamlit_app.py
import streamlit as st
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor

# Загрузка данных

import os
from google.colab import drive
drive.mount('/content/drive') #вылезет регистрация
# Путь к папке с ноутбуком
notebook_path = '/content/drive/MyDrive/доп прога/Шифт интенсив'
# Чтение файла
target = pd.read_csv(os.path.join(notebook_path, 'train_target.csv'))
# Путь к папке с ноутбуком2
notebook_path2 = '/content/drive/MyDrive/доп прога/Шифт интенсив/data'
# Чтение файла
maindf = pd.read_parquet(os.path.join(notebook_path2, 'train_main_df.parquet'))
main_test = pd.read_parquet(os.path.join(notebook_path2, 'test_main_df.parquet'))

feat6 = ['savings_sum_sa_now', 'savings_sum_sa_1m', 'savings_sum_sa_2m', 'savings_sum_sa_3m',
        'savings_avg_sa_3m', 'savings_sum_sa_credit_3m']

train_df = maindf[feat6].copy()
test_df = main_test[feat6].copy()


# Подготовка данных
y = np.log1p(target['target'])
X_train, X_test, y_train, y_test = train_test_split(train_df, y, test_size=0.1, random_state=42)

# Модель
modelcat = CatBoostRegressor(
    bootstrap_type='MVS',
    grow_policy='Lossguide',
    max_leaves=64,
    l2_leaf_reg=5,
    min_data_in_leaf=20,
    iterations=250,
    learning_rate=0.07,
    random_state=42,
    verbose=0,  # Изменили на 0 чтобы не засорять вывод
    rsm=0.7,
    max_depth=6
)

modelcat.fit(X_train, y_train, eval_set=(X_test, y_test), use_best_model=True)

# Функция для предсказания
def predict_balance(sum_now, sum1, sum2, sum3, avg3, credit3):
    input_data = pd.DataFrame([[sum_now, sum1, sum2, sum3, avg3, credit3]],
                             columns=feat6)  # Исправлено на feat6
    prediction = np.expm1(modelcat.predict(input_data))[0]  # Обратное преобразование из log1p
    return round(prediction, 2)

# Интерфейс
st.title('Прогнозирование остатка на счёте')
st.write('Это приложение прогнозирует остаток на счёте на горизонте 2мес')

# Сайдбар с параметрами
st.sidebar.header('Данные клиента')
sum_now = st.sidebar.slider('Текущий остаток (руб)', 0.0, 300000.0, 500.0)
sum1 = st.sidebar.slider('1 мес. назад (руб)', 0.0, 300000.0, 500.0)
sum2 = st.sidebar.slider('2 мес. назад (руб)', 0.0, 300000.0, 500.0)
sum3 = st.sidebar.slider('3 мес. назад (руб)', 0.0, 300000.0, 500.0)
avg3 = st.sidebar.slider('Средний баланс 3 мес. (руб)', 0.0, 300000.0, 500.0)
credit3 = st.sidebar.slider('Поступления за 3 мес. (руб)', 0.0, 500000.0, 500.0)

# Предсказание
if st.sidebar.button('Составить прогноз'):
    prediction = predict_balance(sum_now, sum1, sum2, sum3, avg3, credit3)
    st.subheader('Результат прогноза')
    st.success(f'Прогнозируемый остаток: **{prediction:,} руб**'.replace(",", " "))

# Информация о данных
if st.checkbox('Показать статистику данных'):
    st.subheader('Статистика по признакам')
    st.write(train_df.describe())

Overwriting money_streamlit_app.py


In [ ]:
!streamlit run money_streamlit_app.py




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.74.22.103:8501

  Stopping...
  Stopping...


In [ ]:
!wget https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip
!unzip ngrok-stable-linux-amd64.zip
!streamlit run money_streamlit_app.py &>/dev/null&
!./ngrok http 8501

--2025-07-09 06:58:25--  https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip
Resolving bin.equinox.io (bin.equinox.io)... 35.71.179.82, 99.83.220.108, 75.2.60.68, ...
Connecting to bin.equinox.io (bin.equinox.io)|35.71.179.82|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 13921656 (13M) [application/octet-stream]
Saving to: ‘ngrok-stable-linux-amd64.zip’

ngrok-stable-linux- 100%[===================>]  13.28M  66.6MB/s    in 0.2s    

2025-07-09 06:58:25 (66.6 MB/s) - ‘ngrok-stable-linux-amd64.zip’ saved [13921656/13921656]

Archive:  ngrok-stable-linux-amd64.zip
  inflating: ngrok                   
Usage of ngrok requires a verified account and authtoken.

Sign up for an account: https://dashboard.ngrok.com/signup
Install your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken

ERR_NGROK_4018



# Запуск 3

In [ ]:
!pip install streamlit scikit-learn pandas catboost

In [ ]:
%%writefile money_streamlit_app.py

# Добавьте в начало файла:
import streamlit as st
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor

# Загрузка данных

import os
from google.colab import drive
drive.mount('/content/drive') #вылезет регистрация
# Путь к папке с ноутбуком
notebook_path = '/content/drive/MyDrive/доп прога/Шифт интенсив'
# Чтение файла
target = pd.read_csv(os.path.join(notebook_path, 'train_target.csv'))
# Путь к папке с ноутбуком2
notebook_path2 = '/content/drive/MyDrive/доп прога/Шифт интенсив/data'
# Чтение файла
maindf = pd.read_parquet(os.path.join(notebook_path2, 'train_main_df.parquet'))
main_test = pd.read_parquet(os.path.join(notebook_path2, 'test_main_df.parquet'))

feat6 = ['savings_sum_sa_now', 'savings_sum_sa_1m', 'savings_sum_sa_2m', 'savings_sum_sa_3m',
        'savings_avg_sa_3m', 'savings_sum_sa_credit_3m']

train_df = maindf[feat6].copy()
test_df = main_test[feat6].copy()


# Подготовка данных
y = np.log1p(target['target'])
X_train, X_test, y_train, y_test = train_test_split(train_df, y, test_size=0.1, random_state=42)

# Модель
modelcat = CatBoostRegressor(
    bootstrap_type='MVS',
    grow_policy='Lossguide',
    max_leaves=64,
    l2_leaf_reg=5,
    min_data_in_leaf=20,
    iterations=250,
    learning_rate=0.07,
    random_state=42,
    verbose=0,  # Изменили на 0 чтобы не засорять вывод
    rsm=0.7,
    max_depth=6
)

modelcat.fit(X_train, y_train, eval_set=(X_test, y_test), use_best_model=True)

# Функция для предсказания
def predict_balance(sum_now, sum1, sum2, sum3, avg3, credit3):
    input_data = pd.DataFrame([[sum_now, sum1, sum2, sum3, avg3, credit3]],
                             columns=feat6)  # Исправлено на feat6
    prediction = np.expm1(modelcat.predict(input_data))[0]  # Обратное преобразование из log1p
    return round(prediction, 2)

# Интерфейс
st.title('Прогнозирование остатка на счёте')
st.write('Это приложение прогнозирует остаток на счёте на горизонте 2мес')

# Сайдбар с параметрами
st.sidebar.header('Данные клиента')
sum_now = st.sidebar.slider('Текущий остаток (руб)', 0.0, 300000.0, 500.0)
sum1 = st.sidebar.slider('1 мес. назад (руб)', 0.0, 300000.0, 500.0)
sum2 = st.sidebar.slider('2 мес. назад (руб)', 0.0, 300000.0, 500.0)
sum3 = st.sidebar.slider('3 мес. назад (руб)', 0.0, 300000.0, 500.0)
avg3 = st.sidebar.slider('Средний баланс 3 мес. (руб)', 0.0, 300000.0, 500.0)
credit3 = st.sidebar.slider('Поступления за 3 мес. (руб)', 0.0, 500000.0, 500.0)

# Предсказание
if st.sidebar.button('Составить прогноз'):
    prediction = predict_balance(sum_now, sum1, sum2, sum3, avg3, credit3)
    st.subheader('Результат прогноза')
    st.success(f'Прогнозируемый остаток: **{prediction:,} руб**'.replace(",", " "))

# Информация о данных
if st.checkbox('Показать статистику данных'):
    st.subheader('Статистика по признакам')
    st.write(train_df.describe())

Overwriting money_streamlit_app.py


In [ ]:
# 1. Остановка предыдущих процессов
!pkill -f streamlit

# 2. Запуск на альтернативном порту
!nohup streamlit run --server.port 8000 money_streamlit_app.py &>/dev/null&

# 3. Пауза для инициализации
import time
time.sleep(30)

# 4. Получение ссылки
from google.colab.output import eval_js
try:
    url = eval_js("google.colab.kernel.proxyPort(8000)")
    print(f"Откройте приложение по ссылке: {url}")
except:
    print("Ошибка! Попробуйте:")
    print("1. Перезапустить runtime (Ctrl+M .)")
    print("2. Проверить нет ли ошибок в приложении")
    print("3. Попробовать порт 8502")

Откройте приложение по ссылке: https://8000-m-s-27gcjqxiqk11t-c.us-east1-1.prod.colab.dev


In [ ]:
# Замените &>/dev/null на перенаправление в файл
!nohup streamlit run --server.port 8000 money_streamlit_app.py &> streamlit.log &

# Затем можно посмотреть логи
!cat streamlit.log

In [ ]:
# Явно проверить доступность порта
!netstat -tulnp | grep 8501

# Или попробовать другой порт
!streamlit run --server.port 8000 money_streamlit_app.py



2025-07-09 08:16:32.778 Port 8000 is already in use


In [ ]:
!cat nohup.out

cat: nohup.out: No such file or directory


In [ ]:
!curl -s http://localhost:8000

<!--
 Copyright (c) Streamlit Inc. (2018-2022) Snowflake Inc. (2022-2025)

 Licensed under the Apache License, Version 2.0 (the "License");
 you may not use this file except in compliance with the License.
 You may obtain a copy of the License at

     http://www.apache.org/licenses/LICENSE-2.0

 Unless required by applicable law or agreed to in writing, software
 distributed under the License is distributed on an "AS IS" BASIS,
 WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
 See the License for the specific language governing permissions and
 limitations under the License.
-->

<!DOCTYPE html>
<html lang="en">
  <head>
    <meta charset="UTF-8" />
    <meta
      name="viewport"
      content="width=device-width, initial-scale=1, shrink-to-fit=no"
    />
    <link rel="shortcut icon" href="./favicon.png" />
    <link
      rel="preload"
      href="./static/media/SourceSansVF-Upright.ttf.BsWL4Kly.woff2"
      as="font"
      type="font/woff2"
      crossorig